In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("../data/tarkwa_nasa_power.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("\nColumns:", list(df.columns))
print("\nMissing values:\n", df.isnull().sum())
print("\nSummary stats:")
df.describe().round(2)

Shape: (13328, 10)
Date range: 1990-01-01 to 2026-06-28

Columns: ['date', 'rainfall_mm', 'temp_mean_C', 'temp_max_C', 'temp_min_C', 'humidity_pct', 'pressure_kPa', 'wind_speed_ms', 'year', 'month']

Missing values:
 date             0
rainfall_mm      0
temp_mean_C      0
temp_max_C       0
temp_min_C       0
humidity_pct     0
pressure_kPa     0
wind_speed_ms    0
year             0
month            0
dtype: int64

Summary stats:


,date,rainfall_mm,temp_mean_C,temp_max_C,temp_min_C,humidity_pct,pressure_kPa,wind_speed_ms,year,month
count,13328,13328.00,13328.00,13328.00,13328.00,13328.00,13328.00,13328.00,13328.00,13328.00
mean,2008-03-30 12:00:00,4.48,25.88,30.19,22.65,84.19,99.98,1.10,2007.75,6.48
min,1990-01-01 00:00:00,0.00,20.90,23.99,14.60,35.30,99.45,0.35,1990.00,1.00
25%,1999-02-14 18:00:00,0.86,24.82,28.54,21.93,80.36,99.86,0.88,1999.00,3.00
50%,2008-03-30 12:00:00,2.67,25.83,29.75,22.80,87.24,99.97,1.07,2008.00,6.00
75%,2017-05-14 06:00:00,6.02,26.78,31.41,23.53,89.90,100.10,1.29,2017.00,9.00
max,2026-06-28 00:00:00,137.20,31.24,38.54,26.23,96.36,100.51,2.27,2026.00,12.00
std,NaN,5.70,1.43,2.31,1.28,8.22,0.17,0.28,10.54,3.45


In [2]:
# Aggregate daily data to monthly
# rainfall_mm → sum (monthly total)
# all other meteorological variables → mean (monthly average)
monthly_df = df.groupby(["year", "month"]).agg(
    rainfall_mm=("rainfall_mm", "sum"),
    temp_mean_C=("temp_mean_C", "mean"),
    temp_max_C=("temp_max_C", "mean"),
    temp_min_C=("temp_min_C", "mean"),
    humidity_pct=("humidity_pct", "mean"),
    pressure_kPa=("pressure_kPa", "mean"),
    wind_speed_ms=("wind_speed_ms", "mean"),
).reset_index()

# Add full month names and proper date column
month_names = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}
monthly_df["month_name"] = monthly_df["month"].map(month_names)
monthly_df["date"] = pd.to_datetime(dict(year=monthly_df["year"], month=monthly_df["month"], day=1))
monthly_df = monthly_df.sort_values("date").reset_index(drop=True)

print("Monthly shape:", monthly_df.shape)
print("Date range:", monthly_df["date"].min().date(), "to", monthly_df["date"].max().date())
print("\nFirst 5 rows:")
print(monthly_df.head())
print("\nMonthly rainfall stats:")
print(monthly_df["rainfall_mm"].describe().round(2))

Monthly shape: (438, 11)
Date range: 1990-01-01 to 2026-06-01

First 5 rows:
   year  month  rainfall_mm  temp_mean_C  temp_max_C  temp_min_C  \
0  1990      1        36.92    27.487419   33.637419   22.950323   
1  1990      2        98.45    28.083929   34.751786   23.053929   
2  1990      3        70.49    28.695806   34.812581   24.168710   
3  1990      4       251.53    26.527333   30.549000   23.508667   
4  1990      5       187.32    25.471613   28.980000   22.756129   

   humidity_pct  pressure_kPa  wind_speed_ms month_name       date  
0     70.515161     99.846129       1.018710    January 1990-01-01  
1     69.375714     99.863571       1.145714   February 1990-02-01  
2     71.607097     99.846774       1.170000      March 1990-03-01  
3     86.065333     99.836667       1.134000      April 1990-04-01  
4     90.063226    100.010323       1.042258        May 1990-05-01  

Monthly rainfall stats:
count    438.00
mean     136.41
std       93.68
min        6.31
25%       6

In [3]:
OUTPUT_PATH = "../data/tarkwa_nasa_monthly.csv"
monthly_df.to_csv(OUTPUT_PATH, index=False)

# Reload check
check = pd.read_csv(OUTPUT_PATH, parse_dates=["date"])
print(f"Saved {len(monthly_df)} rows to {OUTPUT_PATH}")
print("Reloaded shape:", check.shape)
print("Reloaded columns:", list(check.columns))

Saved 438 rows to ../data/tarkwa_nasa_monthly.csv
Reloaded shape: (438, 11)
Reloaded columns: ['year', 'month', 'rainfall_mm', 'temp_mean_C', 'temp_max_C', 'temp_min_C', 'humidity_pct', 'pressure_kPa', 'wind_speed_ms', 'month_name', 'date']
